### Sections
| | |
|---|---|
| **A** | Setup & load results |
| **B** | Experiment tracking with MLflow |
| **C** | Hallucination guardrail |
| **D** | Responsible AI card |
| **E** | Summary |


In [ ]:
import json, sys, warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mlflow

warnings.filterwarnings('ignore')

root = Path.cwd().parent
sys.path.insert(0, str(root))
from config import DATA_DIR

OUTPUT_DIR = DATA_DIR / 'outputs'

RESULTS_FILE = OUTPUT_DIR / '03_03_rag_results.json'
if not RESULTS_FILE.exists():
    raise FileNotFoundError('03_03_rag_results.json not found. Run 03_03 first.')

with open(RESULTS_FILE) as f:
    data = json.load(f)

meta    = data['run_metadata']
results = data['query_results']
evals   = data['evaluation']

print(f'Loaded {len(results)} query results.')
print(f' Model   : {meta["groq_model"]}')
print(f' Prompt  : {meta["prompt_strategy"]}')

### Section B — Experiment Tracking with MLflow

Log each business query as a separate MLflow run — capturing the inputs, outputs, latency, and quality scores.

In [ ]:
mlflow.set_experiment('supply_chain_rag')

run_ids = []
for r, e in zip(results, evals):
    with mlflow.start_run(run_name=r['id']):
        # Parameters — what settings were used
        mlflow.log_param('query_id',       r['id'])
        mlflow.log_param('domain',         r['domain'])
        mlflow.log_param('groq_model',     meta['groq_model'])
        mlflow.log_param('prompt_strategy',meta['prompt_strategy'])
        mlflow.log_param('n_results',      meta['n_results'])

        # Metrics — measurable outcomes
        mlflow.log_metric('latency_s',     r['latency_s'])
        mlflow.log_metric('word_count',    r['word_count'])
        mlflow.log_metric('quality_total', e['Total'])
        mlflow.log_metric('quality_pct',   e['Pct'])
        mlflow.log_metric('groundedness',  e['Groundedness'])
        mlflow.log_metric('specificity',   e['Specificity'])
        mlflow.log_metric('actionability', e['Actionability'])
        mlflow.log_metric('structure',     e['Structure'])
        mlflow.log_metric('top1_sim',      r['source_docs'][0]['similarity'] if r['source_docs'] else 0)

        # Artefact — the actual answer text
        answer_file = OUTPUT_DIR / f"answer_{r['id']}.txt"
        answer_file.write_text(f"Q: {r['question']}\n\nA: {r['answer']}")
        mlflow.log_artifact(str(answer_file))

        run_ids.append(mlflow.active_run().info.run_id)

print(f'Logged {len(run_ids)} MLflow runs.')
print(f' Experiment : supply_chain_rag')
print(f' Run IDs    : {[r[:8]+"..." for r in run_ids]}')

In [ ]:
# Pull logged metrics back into a dataframe for display
client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name('supply_chain_rag')
runs   = client.search_runs(exp.experiment_id, order_by=['attributes.start_time'])

log_rows = []
for run in runs:
    m = run.data.metrics
    p = run.data.params
    log_rows.append({
        'Run'        : p.get('query_id','?'),
        'Domain'     : p.get('domain','?')[:25],
        'Latency(s)' : round(m.get('latency_s', 0), 2),
        'Words'      : int(m.get('word_count', 0)),
        'Top-1 Sim'  : round(m.get('top1_sim', 0), 3),
        'Quality %'  : round(m.get('quality_pct', 0), 1),
    })

df_log = pd.DataFrame(log_rows)
print('MLflow Experiment Summary:')
print(df_log.to_string(index=False))

### Section C — Hallucination Guardrail

A lightweight grounding check: for each answer we verify that key noun phrases appear in the retrieved context. If the answer contains claims not found in any retrieved chunk, it is flagged as potentially hallucinated.

In [ ]:
import re

def grounding_check(answer: str, source_docs: list) -> dict:
    """
    Simple grounding check.
    Extracts capitalised noun phrases from the answer and checks
    whether each appears in at least one retrieved source chunk.
    Returns a grounding score and any ungrounded phrases.
    """
    context_text = ' '.join(d['chunk'].lower() for d in source_docs)

    # Extract candidate claims: words/phrases that look like entity names
    candidates = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', answer)
    candidates = list(set(c for c in candidates if len(c) > 4))

    grounded   = [c for c in candidates if c.lower() in context_text]
    ungrounded = [c for c in candidates if c.lower() not in context_text]

    score = len(grounded) / len(candidates) if candidates else 1.0
    return {
        'total_claims'  : len(candidates),
        'grounded'      : len(grounded),
        'ungrounded'    : len(ungrounded),
        'grounding_score': round(score, 3),
        'flagged_phrases': ungrounded[:5],
        'status'        : 'PASS' if score >= 0.6 else 'REVIEW',
    }


print('Grounding Check Results:')
print('-' * 60)
guard_rows = []
for r in results:
    chk = grounding_check(r['answer'], r['source_docs'])
    guard_rows.append({'ID': r['id'], 'Domain': r['domain'][:25], **chk})
    status_icon = 'Y' if chk['status'] == 'PASS' else 'N'
    print(f"{status_icon} {r['id']} | score={chk['grounding_score']} "
          f"| grounded={chk['grounded']}/{chk['total_claims']} "
          f"| status={chk['status']}")
    if chk['flagged_phrases']:
        print(f"   Flagged: {chk['flagged_phrases']}")

df_guard = pd.DataFrame(guard_rows)
avg_score = df_guard['grounding_score'].mean()
print(f'\n   Average grounding score : {avg_score:.3f}')
print(f'   Queries flagged for review : {(df_guard["status"]=="REVIEW").sum()}')

In [ ]:
# Visualise grounding scores
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Hallucination Guardrail — Grounding Analysis', fontweight='bold')

colors = ['#7ED321' if s == 'PASS' else '#E74C3C' for s in df_guard['status']]
axes[0].bar(df_guard['ID'], df_guard['grounding_score'], color=colors, edgecolor='white')
axes[0].axhline(0.6, color='red', linestyle='--', label='Review threshold (0.6)')
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Grounding Score')
axes[0].set_title('Grounding Score per Query')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_guard['grounding_score']):
    axes[0].text(i, v + 0.02, str(v), ha='center', fontsize=9)

axes[1].bar(df_guard['ID'],
            df_guard['grounded'],   color='#7ED321', label='Grounded',   edgecolor='white')
axes[1].bar(df_guard['ID'],
            df_guard['ungrounded'], color='#E74C3C', label='Ungrounded', edgecolor='white',
            bottom=df_guard['grounded'])
axes[1].set_ylabel('Claim Count')
axes[1].set_title('Grounded vs Ungrounded Claims')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_01_grounding.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grounding chart saved.')

### Section D — Responsible AI Card

A Responsible AI card documents the system's design decisions, limitations, and mitigation strategies — standard practice before deploying any AI system to business users.

In [ ]:
rai_card = {
    'System'      : 'Supply Chain RAG Pipeline',
    'Model'       : meta['groq_model'],
    'Version'     : '1.0 (Capstone)',
    'Intended Use': 'Decision support for supply chain analysts. '
                    'Answers questions about suppliers, routes, warehouses, orders.',
    'Out of Scope': 'Real-time data, financial advice, legal decisions, '
                    'fully autonomous actions without human review.',
    'Data'        : 'Synthetic supply chain data. No real personal or '
                    'commercially sensitive data used.',
    'Limitations' : [
        'Answers are only as good as the Knowledge Graph — gaps in KG = gaps in answers.',
        'Fine-tuned embeddings showed mixed results; base model may retrieve more reliably.',
        'Grounding check is heuristic — not a formal hallucination detector.',
        'LLM may occasionally produce plausible-sounding but incorrect inferences.',
    ],
    'Mitigations' : [
        'All answers cite source context entries — reviewable by the analyst.',
        'Grounding score flags answers below 0.6 for human review.',
        'System prompt constrains LLM to answer only from provided context.',
        'Temperature set to 0.1 for low randomness and consistent outputs.',
    ],
    'Bias Considerations': [
        'Supplier rankings are based on synthetic reliability scores — not real-world data.',
        'Route and warehouse recommendations reflect the synthetic data distribution.',
        'No demographic or protected-class data is used in any decision.',
    ],
    'Human Oversight': 'Required. This system is a decision-support tool only. '
                       'All recommendations must be reviewed by a qualified analyst '
                       'before action is taken.',
}

print('=' * 60)
print('  RESPONSIBLE AI CARD')
print('=' * 60)
for key, val in rai_card.items():
    if isinstance(val, list):
        print(f'\n  {key}:')
        for item in val:
            print(f'    • {item}')
    else:
        print(f'\n  {key}:\n    {val}')
print('\n' + '=' * 60)

In [ ]:
# Save Responsible AI card as JSON
rai_file = OUTPUT_DIR / '04_01_responsible_ai_card.json'
with open(rai_file, 'w') as f:
    json.dump(rai_card, f, indent=2)
print(f'Responsible AI card saved → {rai_file}')

In [ ]:
# Visual RAI summary — simple table
fig, ax = plt.subplots(figsize=(13, 6))
ax.axis('off')
fig.suptitle('Responsible AI Summary — Supply Chain RAG Pipeline',
             fontsize=13, fontweight='bold', y=0.98)

sections = [
    ('Intended Use',       'Decision support for supply chain analysts'),
    ('Out of Scope',       'Real-time data · financial/legal decisions · autonomous actions'),
    ('Data',               'Synthetic only — no real personal or sensitive data'),
    ('Hallucination Guard','Grounding score flags low-confidence answers for human review'),
    ('Bias',               'Synthetic data only — no protected-class attributes used'),
    ('Transparency',       'Every answer cites source context entries'),
    ('Human Oversight',    'REQUIRED — analyst must review before action'),
    ('Temperature',        '0.1 — low randomness for consistent outputs'),
]

table_data  = [[s[0], s[1]] for s in sections]
col_labels  = ['Dimension', 'Status / Detail']
row_colors  = [['#EBF5FB','#EBF5FB'] if i % 2 == 0 else ['#FDFEFE','#FDFEFE']
               for i in range(len(table_data))]

tbl = ax.table(
    cellText=table_data, colLabels=col_labels,
    cellLoc='left', loc='center',
    cellColours=row_colors,
    colWidths=[0.25, 0.65],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 2)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2E86C1')
        cell.set_text_props(color='white', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_01_rai_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('RAI summary chart saved.')

### Section E — Project Summary

In [ ]:
print('=' * 65)

print(f'\n  MLflow Tracking')
print(f'    Runs logged   : {len(run_ids)}')
print(f'    Experiment    : supply_chain_rag')

print(f'\n  Hallucination Guardrail')
print(f'    Avg grounding score : {df_guard["grounding_score"].mean():.3f}')
print(f'    Queries PASS        : {(df_guard["status"]=="PASS").sum()} / {len(df_guard)}')

print(f'\n  Responsible AI Card')
print(f'    Dimensions covered  : {len(sections)}')

print(f'\n  Artefacts saved:')
for a in [
    '04_01_grounding.png         — hallucination guardrail chart',
    '04_01_rai_summary.png       — responsible AI table (PPT slide)',
    '04_01_responsible_ai_card.json — full RAI card',
]:
    print(f' {a}')

print('\n' + '=' * 65)